# Code 8.3.1: Exact and near-duplicate detection

Builds five toy documents, one a whitespace variant and one a one-word variant of another. Exact deduplication hashes a normalized copy of each document; near-duplicate detection compares word 3-gram sets by their Jaccard index and estimates it with a 128-hash MinHash signature. The MinHash uses salted MD5 hashes for clarity; production implementations use fast non-cryptographic hash families and locality-sensitive hashing to avoid comparing every pair of signatures.


In [ ]:
import hashlib, re
import torch
import tiktoken

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token                                     # <|endoftext|>, id 50256
page = ("The museum is open from nine in the morning until five in the evening, "
        "and admission is free for children under twelve and for students with a valid card.")
docs = [page,
        "Pretraining data is measured in tokens.",
        page.replace(" ", "  "),                          # duplicate up to whitespace
        page.replace("five", "six"),                      # near duplicate: one word changed
        "A short one."]

# Exact deduplication: hash a normalized copy of each document
def normalize(s):
    return re.sub(r"\s+", " ", s.lower()).strip()
seen, unique = set(), []
for doc in docs:
    key = hashlib.sha256(normalize(doc).encode()).hexdigest()
    if key not in seen:
        seen.add(key); unique.append(doc)
print(len(docs), "->", len(unique), "documents after exact dedup")

# Near-duplicate detection: Jaccard similarity of word 3-gram sets, estimated with MinHash
def shingles(s, k=3):
    w = re.findall(r"\w+", s.lower())
    return {" ".join(w[i:i + k]) for i in range(len(w) - k + 1)}
def minhash(sh, num_hashes=128):
    return [min(int(hashlib.md5(f"{i}:{g}".encode()).hexdigest()[:8], 16) for g in sh)
            for i in range(num_hashes)]
a, b = shingles(unique[0]), shingles(unique[2])
jaccard = len(a & b) / len(a | b)
ma, mb = minhash(a), minhash(b)
estimate = sum(x == y for x, y in zip(ma, mb)) / len(ma)
print(f"Jaccard {jaccard:.2f}, MinHash estimate {estimate:.2f}")
# 5 -> 4 documents after exact dedup
# Jaccard 0.80, MinHash estimate 0.83
